# 🔬 Transfer Learning: Fine-Tuning Fake News BERT Detector

This notebook fine-tunes the pre-trained **[jy46604790/Fake-News-Bert-Detect](https://huggingface.co/jy46604790/Fake-News-Bert-Detect)** model (RoBERTa-based) on your new 20,000+ article dataset using **transfer learning** with:

- **Layer freezing** — freeze early transformer layers to preserve learned representations
- **Hyperparameter tuning** — experiment with learning rate, batch size, warmup, weight decay
- **Gradual unfreezing** — progressive unfreezing for optimal fine-tuning

### Model Info
- **Base model**: `roberta-base` fine-tuned on 40K+ news articles (data up to 2020)
- **Labels**: `LABEL_0` = Fake, `LABEL_1` = Real
- **Max tokens**: 512

---
## 1. Install Dependencies

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn torch pandas matplotlib seaborn

---
## 2. Imports & Device Setup

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## 3. Load & Explore the Dataset

In [ ]:
# ============================================================
# UPDATE THIS PATH:
# - Google Colab: Upload file and use '/content/fake_news_dataset.csv'
# - Local Jupyter: Use the full path to your CSV file
# ============================================================
DATASET_PATH = 'fake_news_dataset.csv'

df = pd.read_csv(DATASET_PATH)
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nLabel distribution:\n{df['label'].value_counts()}")
print(f"\nSample data:")
df.head(3)

In [ ]:
# Visualize label distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Label distribution
colors = ['#e74c3c', '#2ecc71']
df['label'].value_counts().plot(kind='bar', ax=axes[0], color=colors)
axes[0].set_title('Label Distribution (Fake vs Real)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Text length distribution
df['text_length'] = df['text'].astype(str).apply(len)
df['text_length'].hist(bins=50, ax=axes[1], color='#3498db', edgecolor='white')
axes[1].set_title('Text Length Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Character Count')
axes[1].set_ylabel('Frequency')
axes[1].axvline(x=df['text_length'].median(), color='red', linestyle='--', label=f"Median: {df['text_length'].median():.0f}")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nText length stats:")
print(df['text_length'].describe())

---
## 4. Data Preprocessing

In [ ]:
# Clean and prepare data
df = df.dropna(subset=['text', 'label'])
df['text'] = df['text'].astype(str).str.strip()

# Combine title + text for richer context
df['input_text'] = df['title'].fillna('').astype(str) + ' [SEP] ' + df['text'].astype(str)

# Map labels: the model uses LABEL_0 = Fake (0), LABEL_1 = Real (1)
label_map = {'fake': 0, 'real': 1}
df['label_id'] = df['label'].str.lower().str.strip().map(label_map)

# Drop any rows where label mapping failed
df = df.dropna(subset=['label_id'])
df['label_id'] = df['label_id'].astype(int)

print(f"Final dataset size: {len(df)}")
print(f"Label distribution after cleaning:\n{df['label_id'].value_counts().rename({0: 'Fake', 1: 'Real'})}")

In [ ]:
# Train/Validation/Test split (80% / 10% / 10%)
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label_id']
)
val_df, test_df = train_test_split(
    test_df, test_size=0.5, random_state=42, stratify=test_df['label_id']
)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"\nTrain label dist: {dict(train_df['label_id'].value_counts())}")
print(f"Val label dist:   {dict(val_df['label_id'].value_counts())}")
print(f"Test label dist:  {dict(test_df['label_id'].value_counts())}")

---
## 5. Load Pre-trained Model & Tokenizer

In [ ]:
MODEL_NAME = 'jy46604790/Fake-News-Bert-Detect'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

# Print model architecture summary
print("=" * 60)
print("MODEL ARCHITECTURE SUMMARY")
print("=" * 60)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel type: {model.config.model_type}")
print(f"Num hidden layers: {model.config.num_hidden_layers}")
print(f"Hidden size: {model.config.hidden_size}")
print(f"Num labels: {model.config.num_labels}")
print(f"Max position embeddings: {model.config.max_position_embeddings}")

---
## 6. Evaluate Pre-trained Model BEFORE Fine-Tuning (Baseline)

Let's see how the original model performs on our new dataset **before** any fine-tuning. This gives us a baseline to measure improvement.

In [ ]:
from transformers import pipeline

# Create inference pipeline with the original model
clf = pipeline('text-classification', model=MODEL_NAME, tokenizer=MODEL_NAME, device=device, truncation=True)

# Evaluate on a sample of the test set (full test set can take time)
sample_size = min(500, len(test_df))
test_sample = test_df.sample(n=sample_size, random_state=42)

print(f"Evaluating baseline model on {sample_size} test samples...")
baseline_preds = []
for text in test_sample['input_text'].values:
    try:
        result = clf(text[:512])
        pred = int(result[0]['label'].split('_')[-1])  # LABEL_0 -> 0, LABEL_1 -> 1
    except:
        pred = 0
    baseline_preds.append(pred)

baseline_acc = accuracy_score(test_sample['label_id'].values, baseline_preds)
baseline_prec, baseline_rec, baseline_f1, _ = precision_recall_fscore_support(
    test_sample['label_id'].values, baseline_preds, average='weighted'
)

print(f"\n{'='*50}")
print(f"BASELINE MODEL PERFORMANCE (BEFORE FINE-TUNING)")
print(f"{'='*50}")
print(f"Accuracy:  {baseline_acc:.4f}")
print(f"Precision: {baseline_prec:.4f}")
print(f"Recall:    {baseline_rec:.4f}")
print(f"F1-Score:  {baseline_f1:.4f}")

del clf  # Free memory
torch.cuda.empty_cache() if torch.cuda.is_available() else None

---
## 7. 🧊 Layer Freezing Strategy

Freezing layers is crucial in transfer learning:
- **Early layers** capture general language features (grammar, syntax) → **FREEZE these**
- **Later layers** capture task-specific features → **TRAIN these**
- The **classifier head** always gets trained

### Strategy Options:
| Strategy | Layers Frozen | When to Use |
|----------|--------------|-------------|
| Freeze all except classifier | All 12 layers | Very small dataset (<1K) |
| Freeze first 10 layers | Layers 0-9 | Small dataset (1K-5K) |
| Freeze first 8 layers | Layers 0-7 | Medium dataset (5K-10K) |
| **Freeze first 6 layers** | **Layers 0-5** | **Our case (20K+) ✅** |
| Freeze first 4 layers | Layers 0-3 | Large dataset (50K+) |
| No freezing (full fine-tune) | None | Very large dataset (100K+) |

In [ ]:
# ============================================================
# 🔧 CONFIGURABLE: Change this value to experiment!
# Recommended: 6 for ~20K dataset (freeze first 6 of 12 layers)
# ============================================================
NUM_LAYERS_TO_FREEZE = 6


def freeze_model_layers(model, num_layers_to_freeze):
    """
    Freeze the embedding layers and first N transformer layers.
    Keep the last (12 - N) layers and classification head trainable.
    """
    # 1. Freeze ALL embeddings (word, position, token_type, layernorm)
    for param in model.roberta.embeddings.parameters():
        param.requires_grad = False

    # 2. Freeze the first N encoder layers
    for i in range(num_layers_to_freeze):
        for param in model.roberta.encoder.layer[i].parameters():
            param.requires_grad = False

    # 3. Keep remaining layers & classifier head trainable (already True by default)

    # Print summary
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_params = total_params - trainable_params

    print(f"\n{'='*60}")
    print(f"LAYER FREEZING SUMMARY")
    print(f"{'='*60}")
    print(f"Total layers:       12")
    print(f"Frozen layers:      0 to {num_layers_to_freeze - 1} ({num_layers_to_freeze} layers)")
    print(f"Trainable layers:   {num_layers_to_freeze} to 11 ({12 - num_layers_to_freeze} layers) + Classifier")
    print(f"{'─'*60}")
    print(f"Total parameters:     {total_params:>12,}")
    print(f"Frozen parameters:    {frozen_params:>12,} ({frozen_params/total_params*100:.1f}%)")
    print(f"Trainable parameters: {trainable_params:>12,} ({trainable_params/total_params*100:.1f}%)")
    print(f"{'='*60}")

    return model


model = freeze_model_layers(model, NUM_LAYERS_TO_FREEZE)

In [ ]:
# Detailed view: which layers are frozen vs trainable
print("\n📋 Detailed Layer Status:")
print(f"{'Layer':<40} {'Status':<12} {'Parameters':>12}")
print("─" * 65)

# Embeddings
emb_params = sum(p.numel() for p in model.roberta.embeddings.parameters())
emb_trainable = sum(p.numel() for p in model.roberta.embeddings.parameters() if p.requires_grad)
status = '🟢 Trainable' if emb_trainable > 0 else '🔴 Frozen'
print(f"{'Embeddings':<40} {status:<12} {emb_params:>12,}")

# Each encoder layer
for i in range(model.config.num_hidden_layers):
    layer = model.roberta.encoder.layer[i]
    layer_params = sum(p.numel() for p in layer.parameters())
    layer_trainable = sum(p.numel() for p in layer.parameters() if p.requires_grad)
    status = '🟢 Trainable' if layer_trainable > 0 else '🔴 Frozen'
    print(f"{'Encoder Layer ' + str(i):<40} {status:<12} {layer_params:>12,}")

# Classifier head
clf_params = sum(p.numel() for p in model.classifier.parameters())
status = '🟢 Trainable'
print(f"{'Classifier Head':<40} {status:<12} {clf_params:>12,}")
print("─" * 65)

---
## 8. Tokenize the Dataset

In [ ]:
MAX_LENGTH = 512  # RoBERTa max token length

def tokenize_data(df, tokenizer, max_length=MAX_LENGTH):
    """Tokenize a DataFrame into a HuggingFace Dataset."""
    encodings = tokenizer(
        df['input_text'].tolist(),
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors='pt'
    )
    dataset = Dataset.from_dict({
        'input_ids': encodings['input_ids'],
        'attention_mask': encodings['attention_mask'],
        'labels': df['label_id'].values.tolist()
    })
    dataset.set_format('torch')
    return dataset

print("Tokenizing datasets...")
train_dataset = tokenize_data(train_df, tokenizer)
val_dataset = tokenize_data(val_df, tokenizer)
test_dataset = tokenize_data(test_df, tokenizer)

print(f"✅ Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

---
## 9. ⚙️ Hyperparameter Configuration

These are tuned for a ~20K dataset with layer freezing. Adjust as needed.

In [ ]:
# ============================================================
# 🔧 HYPERPARAMETERS — Tune these for best results!
# ============================================================

HYPERPARAMS = {
    'learning_rate':        2e-5,     # Lower LR since we're fine-tuning (try: 1e-5, 2e-5, 3e-5, 5e-5)
    'num_train_epochs':     5,        # Epochs to train (3-10, early stopping will prevent overfitting)
    'per_device_train_batch_size': 16,  # Batch size (8, 16, 32 — reduce if OOM on GPU)
    'per_device_eval_batch_size':  32,  # Eval batch size (can be larger)
    'weight_decay':         0.01,     # L2 regularization (0.01 - 0.1)
    'warmup_ratio':         0.1,      # Warmup steps as ratio of total (0.06 - 0.1)
    'gradient_accumulation_steps': 2, # Effective batch = batch_size × this (increase for larger effective batch)
    'max_grad_norm':        1.0,      # Gradient clipping
    'lr_scheduler_type':    'cosine', # LR schedule: 'linear', 'cosine', 'cosine_with_restarts'
}

print("📋 Hyperparameter Configuration:")
print("─" * 50)
for k, v in HYPERPARAMS.items():
    print(f"  {k:<35} = {v}")
effective_batch = HYPERPARAMS['per_device_train_batch_size'] * HYPERPARAMS['gradient_accumulation_steps']
print(f"  {'effective_batch_size':<35} = {effective_batch}")
print("─" * 50)

---
## 10. Set Up Trainer with Early Stopping

In [ ]:
def compute_metrics(eval_pred):
    """Compute metrics for evaluation."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted'
    )
    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }


# Training arguments
training_args = TrainingArguments(
    output_dir='./fake_news_finetuned',
    num_train_epochs=HYPERPARAMS['num_train_epochs'],
    learning_rate=HYPERPARAMS['learning_rate'],
    per_device_train_batch_size=HYPERPARAMS['per_device_train_batch_size'],
    per_device_eval_batch_size=HYPERPARAMS['per_device_eval_batch_size'],
    weight_decay=HYPERPARAMS['weight_decay'],
    warmup_ratio=HYPERPARAMS['warmup_ratio'],
    gradient_accumulation_steps=HYPERPARAMS['gradient_accumulation_steps'],
    max_grad_norm=HYPERPARAMS['max_grad_norm'],
    lr_scheduler_type=HYPERPARAMS['lr_scheduler_type'],
    
    # Evaluation & saving
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    save_total_limit=3,
    
    # Logging
    logging_dir='./logs',
    logging_steps=50,
    report_to='none',
    
    # Performance
    fp16=torch.cuda.is_available(),  # Mixed precision on GPU
    dataloader_num_workers=2,
    seed=42,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],  # Stop if no improvement for 2 epochs
)

print("✅ Trainer configured with early stopping (patience=2)")

---
## 11. 🚀 Train the Model!

In [ ]:
print("🚀 Starting fine-tuning...")
print(f"   Frozen layers: 0-{NUM_LAYERS_TO_FREEZE - 1}")
print(f"   Trainable layers: {NUM_LAYERS_TO_FREEZE}-11 + Classifier")
print(f"   Epochs: {HYPERPARAMS['num_train_epochs']} (with early stopping)")
print(f"   Learning rate: {HYPERPARAMS['learning_rate']}")
print(f"   Effective batch size: {HYPERPARAMS['per_device_train_batch_size'] * HYPERPARAMS['gradient_accumulation_steps']}")
print("=" * 60)

train_result = trainer.train()

print("\n" + "=" * 60)
print("✅ Training complete!")
print(f"   Total steps: {train_result.global_step}")
print(f"   Training loss: {train_result.training_loss:.4f}")
print("=" * 60)

---
## 12. Plot Training History

In [ ]:
# Extract training history
log_history = trainer.state.log_history

# Separate train and eval logs
train_logs = [log for log in log_history if 'loss' in log and 'eval_loss' not in log]
eval_logs = [log for log in log_history if 'eval_loss' in log]

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 1. Training Loss
if train_logs:
    steps = [log['step'] for log in train_logs]
    losses = [log['loss'] for log in train_logs]
    axes[0].plot(steps, losses, color='#e74c3c', linewidth=2)
    axes[0].set_title('Training Loss', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True, alpha=0.3)

# 2. Eval Loss & Accuracy
if eval_logs:
    epochs = list(range(1, len(eval_logs) + 1))
    eval_losses = [log['eval_loss'] for log in eval_logs]
    eval_accs = [log.get('eval_accuracy', 0) for log in eval_logs]
    eval_f1s = [log.get('eval_f1', 0) for log in eval_logs]
    
    axes[1].plot(epochs, eval_losses, 'o-', color='#e67e22', linewidth=2, markersize=8, label='Eval Loss')
    axes[1].set_title('Validation Loss', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].grid(True, alpha=0.3)
    
    axes[2].plot(epochs, eval_accs, 's-', color='#2ecc71', linewidth=2, markersize=8, label='Accuracy')
    axes[2].plot(epochs, eval_f1s, 'd-', color='#3498db', linewidth=2, markersize=8, label='F1-Score')
    axes[2].set_title('Validation Metrics', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Score')
    axes[2].legend(fontsize=12)
    axes[2].grid(True, alpha=0.3)
    axes[2].set_ylim([0.5, 1.05])

plt.tight_layout()
plt.show()

---
## 13. 📊 Evaluate on Test Set

In [ ]:
# Final evaluation on test set
test_results = trainer.evaluate(test_dataset)

print("\n" + "=" * 60)
print("📊 FINAL TEST SET RESULTS")
print("=" * 60)
print(f"  Accuracy:  {test_results['eval_accuracy']:.4f}")
print(f"  Precision: {test_results['eval_precision']:.4f}")
print(f"  Recall:    {test_results['eval_recall']:.4f}")
print(f"  F1-Score:  {test_results['eval_f1']:.4f}")
print(f"  Loss:      {test_results['eval_loss']:.4f}")
print("=" * 60)

# Compare with baseline
print(f"\n🔄 COMPARISON (Baseline vs Fine-Tuned):")
print(f"  Accuracy:  {baseline_acc:.4f} → {test_results['eval_accuracy']:.4f} ({'+' if test_results['eval_accuracy'] > baseline_acc else ''}{(test_results['eval_accuracy'] - baseline_acc)*100:.2f}%)")
print(f"  F1-Score:  {baseline_f1:.4f} → {test_results['eval_f1']:.4f} ({'+' if test_results['eval_f1'] > baseline_f1 else ''}{(test_results['eval_f1'] - baseline_f1)*100:.2f}%)")

In [ ]:
# Confusion Matrix & Classification Report
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

# Classification Report
print("\n📋 Classification Report:")
print(classification_report(true_labels, preds, target_names=['Fake', 'Real']))

# Confusion Matrix
cm = confusion_matrix(true_labels, preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Fake', 'Real'],
            yticklabels=['Fake', 'Real'],
            annot_kws={'size': 16})
ax.set_xlabel('Predicted', fontsize=14)
ax.set_ylabel('Actual', fontsize=14)
ax.set_title('Confusion Matrix — Fine-Tuned Model', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 14. 💾 Save the Fine-Tuned Model

In [ ]:
SAVE_DIR = './fake_news_finetuned_final'

# Save model and tokenizer
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"✅ Model saved to: {SAVE_DIR}")
print(f"\nFiles saved:")
import os
for f in os.listdir(SAVE_DIR):
    size_mb = os.path.getsize(os.path.join(SAVE_DIR, f)) / (1024*1024)
    print(f"  {f} ({size_mb:.1f} MB)")

---
## 15. 🧪 Test with Custom News Articles

In [ ]:
# Load the fine-tuned model for inference
from transformers import pipeline

finetuned_clf = pipeline(
    'text-classification',
    model=SAVE_DIR,
    tokenizer=SAVE_DIR,
    device=device,
    truncation=True
)

# Test articles
test_articles = [
    {
        'title': 'Real News Example',
        'text': 'The Federal Reserve raised interest rates by 0.25 percentage points on Wednesday, '
                'bringing the benchmark rate to a range of 5.25% to 5.50%. Fed Chair Jerome Powell '
                'stated that the decision was made in response to persistent inflation and a robust labor market.'
    },
    {
        'title': 'Fake News Example',
        'text': 'SHOCKING: Scientists discover that drinking bleach cures all diseases! The government '
                'has been hiding this miracle cure for years. Share this before they delete it! '
                'Big pharma does not want you to know about this incredible discovery.'
    },
    {
        'title': 'Ambiguous Example',
        'text': 'A new study suggests that eating chocolate every day can increase life expectancy by 10 years. '
                'Researchers from an undisclosed university conducted the study on a small sample of volunteers.'
    },
]

label_names = {0: '🔴 FAKE', 1: '🟢 REAL'}

print("\n" + "=" * 60)
print("🧪 PREDICTIONS ON CUSTOM ARTICLES")
print("=" * 60)
for article in test_articles:
    input_text = article['title'] + ' [SEP] ' + article['text']
    result = finetuned_clf(input_text)
    label_id = int(result[0]['label'].split('_')[-1])
    confidence = result[0]['score']
    
    print(f"\n📰 {article['title']}")
    print(f"   Prediction:  {label_names[label_id]}")
    print(f"   Confidence:  {confidence:.4f} ({confidence*100:.1f}%)")
    print(f"   Text:        {article['text'][:100]}...")
    print("─" * 60)

---
## 16. 🧪 (Optional) Gradual Unfreezing — Advanced Strategy

If you want even better results, try **gradual unfreezing**:
1. Train with many layers frozen (e.g., first 10 frozen)
2. Unfreeze 2 more layers, train again with lower LR
3. Repeat until you find the sweet spot

This avoids catastrophic forgetting while still adapting deeper layers.

In [ ]:
# ============================================================
# ⚠️  OPTIONAL: Only run this section if you want to try
#     gradual unfreezing for potentially higher accuracy.
#     This will reload the model and train in 3 phases.
# ============================================================

RUN_GRADUAL_UNFREEZING = False  # 👈 Set to True to run this

if RUN_GRADUAL_UNFREEZING:
    print("🔓 Starting Gradual Unfreezing...")
    
    # Reload fresh pretrained model
    model_gradual = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
    model_gradual.to(device)
    
    phases = [
        {'freeze_layers': 10, 'lr': 3e-5, 'epochs': 2, 'desc': 'Phase 1: Train layers 10-11 + Classifier'},
        {'freeze_layers': 6,  'lr': 1e-5, 'epochs': 2, 'desc': 'Phase 2: Train layers 6-11 + Classifier'},
        {'freeze_layers': 3,  'lr': 5e-6, 'epochs': 2, 'desc': 'Phase 3: Train layers 3-11 + Classifier'},
    ]
    
    for phase in phases:
        print(f"\n{'='*60}")
        print(f"  {phase['desc']}")
        print(f"  LR: {phase['lr']} | Epochs: {phase['epochs']}")
        print(f"{'='*60}")
        
        # Re-freeze appropriate layers
        # First unfreeze everything
        for param in model_gradual.parameters():
            param.requires_grad = True
        # Then freeze as specified
        model_gradual = freeze_model_layers(model_gradual, phase['freeze_layers'])
        
        phase_args = TrainingArguments(
            output_dir=f"./fake_news_gradual_phase",
            num_train_epochs=phase['epochs'],
            learning_rate=phase['lr'],
            per_device_train_batch_size=16,
            per_device_eval_batch_size=32,
            weight_decay=0.01,
            warmup_ratio=0.1,
            eval_strategy='epoch',
            save_strategy='epoch',
            load_best_model_at_end=True,
            metric_for_best_model='f1',
            greater_is_better=True,
            save_total_limit=2,
            logging_steps=50,
            report_to='none',
            fp16=torch.cuda.is_available(),
            seed=42,
        )
        
        phase_trainer = Trainer(
            model=model_gradual,
            args=phase_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=compute_metrics,
        )
        
        phase_trainer.train()
        eval_results = phase_trainer.evaluate(test_dataset)
        print(f"  → Test Accuracy: {eval_results['eval_accuracy']:.4f} | F1: {eval_results['eval_f1']:.4f}")
    
    # Save gradual unfreezing model
    GRADUAL_SAVE_DIR = './fake_news_gradual_final'
    phase_trainer.save_model(GRADUAL_SAVE_DIR)
    tokenizer.save_pretrained(GRADUAL_SAVE_DIR)
    print(f"\n✅ Gradual unfreezing model saved to: {GRADUAL_SAVE_DIR}")
else:
    print("⏭️  Gradual unfreezing skipped. Set RUN_GRADUAL_UNFREEZING = True to try it.")

---
## 17. 📦 (Optional) Push to Hugging Face Hub

In [ ]:
# ============================================================
# Uncomment and fill in to push your model to HuggingFace Hub
# ============================================================

# from huggingface_hub import login
# login(token='YOUR_HF_TOKEN')  # Get token from https://huggingface.co/settings/tokens

# trainer.push_to_hub(
#     repo_id='your-username/fake-news-detector-finetuned',
#     commit_message='Fine-tuned Fake News BERT on 20K+ articles dataset'
# )

---
## 📝 Tips for Better Results

| Tip | How |
|-----|-----|
| **Try different freeze levels** | Change `NUM_LAYERS_TO_FREEZE` (4, 6, 8, 10) and compare |
| **Adjust learning rate** | Lower LR (1e-5) for more layers frozen, higher (5e-5) for fewer |
| **Increase epochs** | Set `num_train_epochs` to 8-10 with early stopping patience=3 |
| **Try gradual unfreezing** | Set `RUN_GRADUAL_UNFREEZING = True` in Section 16 |
| **Use larger batches** | Increase `gradient_accumulation_steps` for larger effective batch |
| **Data augmentation** | Back-translation, synonym replacement, random deletion |
| **Try cosine_with_restarts** | Change `lr_scheduler_type` for cyclic learning |
| **Discriminative LR** | Use different LR per layer (lower for earlier layers) |
| **Ensemble** | Train multiple models with different seeds and average predictions |